# Data Cleaning
`2025-05-25`  
Hola Mariano, espero que no haya pasado mucho tiempo desde que habías visto este código.  
  
El día de hoy haré lo siguiente: 
- Limpiar los datos de accidentes para quedarnos solo con aquellos que fueron reales
- Clasificar los datos por tipo de incidente
    - Minor: sin lesiones
    - PIC: personal injuries (something with C)
    - FCS: fatal crash (something with S other than something)
- Quedarnos solo con datos que correspondan a 3 años antes y 3 años después de la intervención
    - La literatura muestra que esa ventana de tiempo es la óptima a considerar
 
`2025-08-06`  
Hola Mariano, sí pasó un chingo de tiempo :(

In [85]:
import os
import pandas as pd
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString, MultiLineString, Polygon
import warnings

warnings.filterwarnings("ignore")
PATH_DATA = '../data'

In [86]:
INICIO_OPERACIONES = pd.to_datetime("2019-04-22")
FROM_DATE = INICIO_OPERACIONES - pd.Timedelta(days=365*3)
TO_DATE = INICIO_OPERACIONES + pd.Timedelta(days=365*3)

In [3]:
accidentes = (
    pd
    .read_parquet(os.path.join(PATH_DATA, 'incidentes-viales.parquet'))
    .reset_index(drop=True)
    .pipe(
        lambda df: 
        gpd
        .GeoDataFrame(
            data=df.drop(['latitud', 'longitud'], axis=1),
            geometry=gpd.points_from_xy(df.longitud, df.latitud)
        )
    )
)

In [4]:
def assign_incident_level(row) -> str:
    """
    Assign the level of the incident based on the following rules:

    MIN - Aquellos incidentes menores en los que no haya lesionados o fatalidades
    PIC - Incidentes con lesiones personales para algún involucrado
    FCS - Incidentes fatales donde la vida de una o más personas se haya perdido
    """
    tipo_incidente = row["tipo_incidente_c4"]
    if tipo_incidente in ["Cadáver"]:
        return "FCS"

    # Para este punto ya nada más quedan Accidentes y lesionados
    emergency_level = row["clas_con_f_alarma"]
    if emergency_level == "URGENCIAS MEDICAS": # Todos los lesionados están aquí
        return "PIC"
    return "MIN" # Aquí solo quedan accidentes menores

In [5]:
INCIDENT_TYPE = ["Accidente", "Cadáver", "Lesionado"]

classified_incidents = (
    accidentes
    # Nos quedamos solo con la ventana de tiempo a estudiar -> 433,235
    .query("@FROM_DATE <= timestamp <= @TO_DATE")
    # Eliminamos estas categorías: Detención ciudadana, Mi Calle, Mi Taxi, Sismo -> 432,707
    .query("tipo_incidente_c4 in @INCIDENT_TYPE and clas_con_f_alarma != 'FALSA ALARMA'")
    # Y ahora hay que clasificar cada uno de los incidentes
    # MIN    239069
    # PIC    191681
    # FCS      1957
    .assign(
        incident_level=lambda df: df.apply(assign_incident_level, axis=1),
        hour=lambda x: x.timestamp.dt.hour,
        weekday=lambda x: x.timestamp.dt.day_name(),
        before_treatment=lambda x: x.timestamp.apply(lambda t: t < INICIO_OPERACIONES)
    )
)

In [6]:
fig, axes = plt.subplots(2,2, figsize=(20,10))

ax = axes[0][0]
sns.kdeplot(
    data=classified_incidents,
    x='hour',
    hue='incident_level',
    common_norm=False,
    fill=True,
    ax=ax,
    palette=["#cccccc", "#a5a5a5", "#ad2e24"],
    cut=0
)
ax.set_title("Distribución incidentes por hora y nivel")
ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)

ax = axes[0][1]
sns.kdeplot(
    data=classified_incidents,
    x='hour',
    hue='incident_level',
    common_norm=False,
    fill=True,
    ax=ax,
    palette=["#cccccc", "#a5a5a5", "#ad2e24"],
    multiple="fill",
    cut=0
)
ax.set_title("Proporción accidentes por hora y tipo")
ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)


ax = axes[1][0]
desfazado = 7
ticks = [hour%24 for hour in range(desfazado,24+desfazado)]

sns.kdeplot(
    data=classified_incidents.assign(hour=lambda x: (x.hour - desfazado)%24),
    x='hour',
    hue='incident_level',
    common_norm=False,
    fill=True,
    ax=ax,
    palette=["#cccccc", "#a5a5a5", "#ad2e24"],
    cut=0
)
ax.set_title(f"Distribución incidentes por hora y nivel (-{desfazado} horas)")
ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)
ax.set_xticks(range(0,24), ticks)

ax = axes[1][1]
sns.kdeplot(
    data=classified_incidents.assign(hour=lambda x: (x.hour - desfazado)%24),
    x='hour',
    hue='incident_level',
    common_norm=False,
    fill=True,
    ax=ax,
    palette=["#cccccc", "#a5a5a5", "#ad2e24"],
    multiple="fill",
    cut=0
)
ax.set_title(f"Proporción accidentes por hora y tipo (-{desfazado} horas)")
ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)
ax.set_xticks(range(0,24), ticks)


fig.tight_layout()
# fig.savefig(os.path.join(PATH_DATA, "graphs", "hour-level-distribution.png"), dpi=300, transparent=True)
plt.close()

In [7]:
final_incidents = (
    classified_incidents
    .pipe(lambda df: df.join(df.get_coordinates()))
    .rename({"x":"longitude", "y":"latitude"}, axis=1)
    .drop([
        "tipo_incidente_c4", "incidente_c4", "clas_con_f_alarma", "geometry"
    ], axis=1)
    .reset_index(drop=True)
    .astype({
        "incident_level":"category",
        "hour":"int8",
        "weekday":"category"
    })
)
final_incidents.to_parquet(os.path.join(PATH_DATA, "classified-incidents.parquet"), index=False)

`2025-11-10`  
Vamos a limpiar los datos del volumen de tráfico, y lo haré a nivel semanal, de hecho tendré que rehacer la mayor parte del análisis.

In [33]:
volumen = pd.read_csv(os.path.join(PATH_DATA, "volumen-anual-total-radares.csv"))

In [47]:
# Vamos a asignar un timestamp para el mes correspondiente
# Y también vamos a calcular el volumen mensual total para cada uno de los radares
factor = 1_000
save = False

volumen_total_mensual = (
    volumen
    .assign(
        porcentaje=lambda x: x.porcentaje / 100,
        volumen_total_mensual=lambda x: x.porcentaje * x.volumen_total_anual / 1_000,
        timestamp=lambda x: pd.to_datetime(x.apply(lambda r: f"{r.year}-{r.month}-01", axis=1))
    )
    # year y month salen del timestamp
    # volumen total anual sale de sumar por año el volumen total mensual
    # porcentaje sale de dividir el volumen total anual entre cad volumen total mensual
    .drop(columns=["year", "month", "volumen_total_anual", "porcentaje"])
    .groupby("timestamp")
    .agg(volumen_mensual=pd.NamedAgg("volumen_total_mensual", "sum"))
    .reset_index()
)
if save:
    volumen_total_mensual.to_parquet(os.path.join(PATH_DATA, "volumen-total-mensual.parquet"), index=False)

In [84]:
# Vamos a hacer una pequeña gráfica para mostrar cómo se movió el tráfico y que definitivamente hubo una disminución
# durante los años de pandemia
fig, ax = plt.subplots(figsize=(13,5))

ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)
ax.spines.left.set_visible(False)


axis_color = "#495057"
ax.xaxis.label.set_color(axis_color)
ax.spines.bottom.set_color(axis_color)
ax.tick_params(axis='both', colors=axis_color)
ax.grid(axis='y', alpha=.2)
ax.get_xaxis().set_ticks(
    ticks=volumen_total_mensual[volumen_total_mensual.timestamp.dt.month == 1].timestamp,
    labels=volumen_total_mensual[volumen_total_mensual.timestamp.dt.month == 1].timestamp.dt.year
)

color = "#0077b6"
ax.plot(volumen_total_mensual.timestamp, volumen_total_mensual.volumen_mensual, color=color)

ax.vlines(
    x=pd.to_datetime("2019-04-22"), 
    ymin=7_500, ymax=20_000, 
    zorder=-10, color="gray", 
    linestyles="dashed", 
    linewidth=1, alpha=.4
)
ax.annotate(
    "Inicio\nFotocívicas", 
    xy=(pd.to_datetime("2019-05-22"), 19_000), 
    ha="left", 
    va="top", 
    color="gray"
)

ax.set_title("Volumen de tránsito total mensual", color=axis_color)
ax.set_ylabel("Volumen total (miles)", color=axis_color)
fig.tight_layout()

plt.savefig(os.path.join(PATH_DATA, "graphs", "volumen-transito-total-mensual.png"), dpi=300, transparent=True)
plt.close()